In [11]:
import sys
from pathlib import Path

import torch
from torchvision.transforms import v2 as transforms

this_path = Path(__file__) if '__file__' in globals() else Path("<undefined>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

ee_tools_path_p = work_path / Path("ee")
sys.path.append(str(ee_tools_path_p))

import utils
from datasets import fetch_handler
from run_manager import RunManager, RunsManager
from network import Network, Networks
from trainer import MultiTrainer, Trainer

from ee_tools.models.resnet_cifar_ee import resnet18 as resnet18_cifar_ee
from ee_tools.models.resnet_cifar_ee_hase import resnet18 as resnet18_cifar_hase
from models_h.model_factories import create_model_ensembles
from models.resnet_cifar import resnet18 as resnet18_cifar

In [12]:
model_config = {
    "name": "resnet18",
    "num_classes": 100,
    "T": 1,
    "pretrained": False,
    "div": 1,  # is_ee=True の場合、この引数は無視されます
    "ensembles": 16,  # YAMLの 'num_models: 4' に対応
    "for_cifar_customize": True,
    "is_ee": True,
    "is_he": False
}

model = create_model_ensembles(**model_config)

net1 = Network(model)

check:   --- >  resnet18 100 1 False 1 16 True True


In [13]:
# net2 = Network(resnet18_cifar_ee(num_classes=100, channels=16, ensembles=16))
net3 = Network(resnet18_cifar_hase(num_classes=100, channels=16, ensembles=16))


In [14]:
utils.comp_param_stat([net1, net3], layer_width=50)

Network: Network                                                            Network: Network                                                        
------------------------------------------------------------------------    ------------------------------------------------------------------------
Layer                                                    Mean        Var    Layer                                                    Mean        Var
------------------------------------------------------------------------    ------------------------------------------------------------------------
base_model.model.conv1.weight                       -0.001544   0.013939    base_model.conv1.weight                              0.002191   0.013911
base_model.model.bn1.weight                          1.000000   0.000000    base_model.bn1.weight                                1.000000   0.000000
base_model.model.bn1.bias                            0.000000   0.000000    base_model.bn1.bias           

In [15]:
print(net1.torchinfo(input_size=(1, 3, 32, 32)))

Layer (type (var_name))                            Input Shape               Output Shape              Kernel Shape              Param #                   Mult-Adds
EasyEnsembleWrapperV2 (EasyEnsembleWrapperV2)      [1, 3, 32, 32]            [1, 100]                  --                        --                        --
├─ResNet (model)                                   [1, 48, 32, 32]           [1, 2048]                 --                        --                        --
│    └─Conv2d (conv1)                              [1, 48, 32, 32]           [1, 256, 32, 32]          [3, 3]                    6,912                     7,077,888
│    └─BatchNorm2d (bn1)                           [1, 256, 32, 32]          [1, 256, 32, 32]          --                        512                       512
│    └─ReLU (relu)                                 [1, 256, 32, 32]          [1, 256, 32, 32]          --                        --                        --
│    └─Identity (maxpool)            